# README Benchmark Runner

Run the cells top to bottom. Edit only the configuration cell for normal experiment changes.

## Configuration

In [8]:
EXPERIMENT = {
    "run_name": "default_run",
    "data_csv": "data/repo_list.csv",
    "tools": {
        "osa": {"enabled": False},
        "readmeready": {"enabled": False},
        "larch": {"enabled": False, "mode": "local"},
        "opencode": {"enabled": True},
    },
    "models": [
        # "openai/gpt-4.1",
        # "anthropic/claude-sonnet-4",
        "google/gemma-3-27b-it",
    ],
    "judge": {
        "api": "openrouter",
        "base_url": "https://openrouter.ai/api/v1",
        "model": "gpt-4.1",
    },
    "reset": {
        "repos": False,
        "tool_outputs": False,
        "evaluation": True,
    },
    # Set to an integer for smoke tests, or None for the full CSV.
    "limit_repos": None,
}


## Imports And Paths

In [2]:
from pathlib import Path
import importlib
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "benchmark_README", *cwd.parents]
BM = next((p for p in candidates if (p / "src" / "notebook_utils.py").is_file()), cwd)
if str(BM) not in sys.path:
    sys.path.insert(0, str(BM))

import src.notebook_utils as notebook_utils
import src.tool_runners as tool_runners
import src.evaluation as evaluation

importlib.reload(notebook_utils)
importlib.reload(tool_runners)
importlib.reload(evaluation)

from src.notebook_utils import build_paths, ensure_directories, load_env_file, read_repo_table, prepare_repositories
from src.tool_runners import run_selected_tools
from src.evaluation import evaluate_outputs

load_env_file(BM / ".env")
paths = build_paths(EXPERIMENT["run_name"])
ensure_directories(paths)
print(paths)


BenchmarkPaths(benchmark_root=WindowsPath('D:/VKR_README_EVAL/benchmark_README'), workspace_root=WindowsPath('D:/VKR_README_EVAL'), experiment_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run'), repositories_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/repositories'), original_readmes_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/original_readmes'), structures_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/repo_structures'), logs_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/logs'), tools_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/tools'), evaluation_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/evaluation'))


## Load Repository Table

In [3]:
repo_df = read_repo_table(EXPERIMENT["data_csv"])
if EXPERIMENT.get("limit_repos"):
    repo_df = repo_df.head(int(EXPERIMENT["limit_repos"]))
repo_df.head()


,repo_url,repo_name,commit_sha,commit_date,repository,repo_slug
0,https://github.com/AntonOsika/gpt-engineer,AntonOsika/gpt-engineer,a90fcd543eedcc0ff2c34561bc0785d2ba83c47e,2024-11-17T22:42:12+00:00,https://github.com/AntonOsika/gpt-engineer,gpt-engineer
1,https://github.com/THUDM/ChatGLM-6B,THUDM/ChatGLM-6B,401bf3a8a7dd8a26fba189551dccfc61a7079b4e,2024-06-27T04:05:25+00:00,https://github.com/THUDM/ChatGLM-6B,ChatGLM-6B
2,https://github.com/OpenEthan/SMSBoom,OpenEthan/SMSBoom,38ccbeccb05fab66442988b38b559062d7cf92d8,2024-03-20T03:30:32+00:00,https://github.com/OpenEthan/SMSBoom,SMSBoom
3,https://github.com/lra/mackup,lra/mackup,45a9929f1117de2b26e7f6264453eb1546882116,2025-06-10T15:53:03+00:00,https://github.com/lra/mackup,mackup
4,https://github.com/chenfei-wu/TaskMatrix,chenfei-wu/TaskMatrix,4b7664f8d3a23804ac1b795d75a73efd162769f0,2023-06-29T08:44:29+00:00,https://github.com/chenfei-wu/TaskMatrix,TaskMatrix


## Pre-flight: Clone, Checkout, Snapshot

In [4]:
preflight = prepare_repositories(
    repo_df,
    paths,
    reset=bool(EXPERIMENT.get("reset", {}).get("repos", False)),
)
preflight


,repo_url,repo_name,repo_slug,commit_sha,status,started_at,finished_at,repo_dir,original_readme,structure_json,error
0,https://github.com/AntonOsika/gpt-engineer,AntonOsika/gpt-engineer,gpt-engineer,a90fcd543eedcc0ff2c34561bc0785d2ba83c47e,failed,2026-05-26T16:52:03+00:00,2026-05-26T16:52:07+00:00,,,,fatal: unable to read tree (a90fcd543eedcc0ff2...
1,https://github.com/THUDM/ChatGLM-6B,THUDM/ChatGLM-6B,ChatGLM-6B,401bf3a8a7dd8a26fba189551dccfc61a7079b4e,failed,2026-05-26T16:52:07+00:00,2026-05-26T16:52:10+00:00,,,,fatal: unable to read tree (401bf3a8a7dd8a26fb...
2,https://github.com/OpenEthan/SMSBoom,OpenEthan/SMSBoom,SMSBoom,38ccbeccb05fab66442988b38b559062d7cf92d8,done,2026-05-26T16:52:10+00:00,2026-05-26T16:52:11+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
3,https://github.com/lra/mackup,lra/mackup,mackup,45a9929f1117de2b26e7f6264453eb1546882116,done,2026-05-26T16:52:11+00:00,2026-05-26T16:52:12+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
4,https://github.com/chenfei-wu/TaskMatrix,chenfei-wu/TaskMatrix,TaskMatrix,4b7664f8d3a23804ac1b795d75a73efd162769f0,done,2026-05-26T16:52:12+00:00,2026-05-26T16:52:13+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
5,https://github.com/hacksider/Deep-Live-Cam,hacksider/Deep-Live-Cam,Deep-Live-Cam,f0fae811d86d05e8b94f31deec8b5456525f6b7b,done,2026-05-26T16:52:13+00:00,2026-05-26T16:52:14+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
6,https://github.com/google/python-fire,google/python-fire,python-fire,dba7e1d0da014e555d174225fdf5ab4c4574b18b,done,2026-05-26T16:52:14+00:00,2026-05-26T16:52:15+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
7,https://github.com/stitionai/devika,stitionai/devika,devika,3b98ed31e22cd2893563d4b375c2538c72a2e224,done,2026-05-26T16:52:15+00:00,2026-05-26T16:52:17+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
8,https://github.com/Pythagora-io/gpt-pilot,Pythagora-io/gpt-pilot,gpt-pilot,4ab9367875d9a2cfbee3aec4d3e74b4ae544fc96,done,2026-05-26T16:52:17+00:00,2026-05-26T16:52:18+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,
9,https://github.com/CorentinJ/Real-Time-Voice-C...,CorentinJ/Real-Time-Voice-Cloning,Real-Time-Voice-Cloning,440322b8ff93ed1cca1d954f781bcfbe83429d23,done,2026-05-26T16:52:18+00:00,2026-05-26T16:52:19+00:00,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,


## Run Selected Tools

In [5]:
tool_status = run_selected_tools(repo_df, paths, EXPERIMENT)
tool_status.tail(20)


,tool,model,model_label,repo_slug,status,started_at,finished_at,duration_sec,output_path,log_path,error
174,opencode,google/gemma-3-27b-it,gemma-3-27b-it,rembg,failed,2026-05-26T16:55:37+00:00,2026-05-26T16:55:41+00:00,4.227,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
175,opencode,google/gemma-3-27b-it,gemma-3-27b-it,devops-exercises,failed,2026-05-26T16:55:41+00:00,2026-05-26T16:55:46+00:00,5.234,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
176,opencode,google/gemma-3-27b-it,gemma-3-27b-it,MiniGPT-4,failed,2026-05-26T16:55:46+00:00,2026-05-26T16:55:50+00:00,4.233,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
177,opencode,google/gemma-3-27b-it,gemma-3-27b-it,python-spider,failed,2026-05-26T16:55:50+00:00,2026-05-26T16:55:57+00:00,6.237,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
178,opencode,google/gemma-3-27b-it,gemma-3-27b-it,chatgpt-retrieval-plugin,failed,2026-05-26T16:55:57+00:00,2026-05-26T16:56:01+00:00,4.255,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
179,opencode,google/gemma-3-27b-it,gemma-3-27b-it,flux,failed,2026-05-26T16:56:01+00:00,2026-05-26T16:56:05+00:00,4.251,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
180,opencode,google/gemma-3-27b-it,gemma-3-27b-it,EmotiEffLib,failed,2026-05-26T16:56:05+00:00,2026-05-26T16:56:10+00:00,5.246,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
181,opencode,google/gemma-3-27b-it,gemma-3-27b-it,Ride,failed,2026-05-26T16:56:10+00:00,2026-05-26T16:56:16+00:00,5.236,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
182,opencode,google/gemma-3-27b-it,gemma-3-27b-it,delPezzo,failed,2026-05-26T16:56:16+00:00,2026-05-26T16:56:26+00:00,10.235,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...
183,opencode,google/gemma-3-27b-it,gemma-3-27b-it,chatsky,failed,2026-05-26T16:56:26+00:00,2026-05-26T16:56:30+00:00,4.226,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,OpenCode did not create .opencode_benchmark_RE...


## Evaluate Generated READMEs

In [6]:
eval_rows = evaluate_outputs(
    paths,
    EXPERIMENT,
    reset=bool(EXPERIMENT.get("reset", {}).get("evaluation", False)),
)
eval_rows.tail(20)


d:\VKR_README_EVAL\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

,tool,model,model_label,repo_slug,score,reason,success,output_path
63,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,rembg,1.0,The README in the actual output fully aligns w...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
64,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,devops-exercises,0.9,The README content in the actual output is com...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
65,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,MiniGPT-4,1.0,The provided README addresses all evaluation s...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
66,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,python-spider,1.0,The README in the actual output strongly align...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
67,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,chatgpt-retrieval-plugin,1.0,The provided README addresses all evaluation s...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
68,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,flux,1.0,The actual output README fully aligns with all...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
69,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,EmotiEffLib,0.9,The README addresses all evaluation steps: it ...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
70,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,Ride,0.8,The README is present in the repository struct...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
71,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,delPezzo,1.0,The actual README addresses all evaluation ste...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
72,opencode,anthropic/claude-sonnet-4,claude-sonnet-4,chatsky,0.9,The response aligns very strongly with all eva...,True,D:\VKR_README_EVAL\benchmark_README\results\de...


## Summary

In [7]:
summary_path = paths.evaluation_dir / "final_summary.csv"
if summary_path.is_file():
    import pandas as pd
    display(pd.read_csv(summary_path))
else:
    print("No summary yet. Run evaluation after at least one successful tool output.")


,tool,model_label,mean_score,evaluated,rows
0,opencode,claude-sonnet-4,0.952381,42,42
1,opencode,gpt-4.1,0.941026,39,39
2,osa,gpt-4.1,1.000000,2,2
